In [1]:
import os

In [2]:
all_paths=os.listdir(r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec')

In [3]:
all_paths[0]

'0000'

In [11]:
import shutil
for i in all_paths:
    c_path=os.path.join(r'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec', i)
    for j in os.listdir(c_path):
        co_path=os.path.join(c_path, j, j+'.npy')
        os.makedirs(rf'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec_compiled\{i}', exist_ok=True)
        shutil.copy(co_path, rf'C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_vec_compiled\{i}')
        # print(co_path)
    print(i, end='\r')

In [ ]:
from openai import AsyncAzureOpenAI
import base64
import os
import json
from pathlib import Path
import asyncio

# Setup Azure OpenAI client
endpoint = "https://student-hub-01.cognitiveservices.azure.com/"
model_name = "gpt-5-mini"
deployment = "gpt-5-mini"
subscription_key = "ESmBSDqea72HypF9RIVjKqmk7HsJjTVa4j2DdQ1oENQHjaFsbo72JQQJ99CBACYeBjFXJ3w3AAAAACOGrfq8"
api_version = "2024-12-01-preview"

client = AsyncAzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

prompt = (
    "Analyze this vehicle invoice/document and extract the following details.\n"
    "Use the visual information from the image to answer accurately.\n\n"
    "If a specific field is not explicitly labeled, infer it from the context (e.g., if 'HP' is not listed separately, look inside the Model Name).\n\n"
    "Extract these 4 fields:\n"
    "1. *dealer_name*: Name of the dealer/agency at the top.\n"
    "2. *model_name*: The vehicle model description. DO NOT include the horsepower in the model name.\n"
    "3. *horse_power*: The horsepower (USUALLY IN THE RANGE OF 10-60). Look for 'HP' or numbers like '47', '50', '55' near the model name. Return as numeric value only (e.g., 47, not '47 HP').\n"
    "4. *asset_cost*: The final invoice total or 'Grand Total' (USUALLY IN THE RANGE OF 100000-10000000). Return as numeric value only (e.g., 850000, not '8,50,000').\n\n"
    "*Output Format:*\n"
    "Return a valid JSON object only. Do NOT use markdown.\n"
    "{\n"
    "  \"fields\": {\n"
    "    \"dealer_name\": \"...\",\n"
    "    \"model_name\": \"...\",\n"
    "    \"horse_power\": <number>,\n"
    "    \"asset_cost\": <number>\n"
    "  }\n"
    "}"
)

# Directory containing all images
train_dir = r"C:\Users\LEGION\Downloads\train\train"
output_dir = r"C:\Users\LEGION\Downloads\train_predictions"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get all image files (common formats)
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
image_files = []
for ext in image_extensions:
    image_files.extend(Path(train_dir).glob(f'*{ext}'))
    image_files.extend(Path(train_dir).glob(f'*{ext.upper()}'))

print(f"Found {len(image_files)} images to process")

# Async function to process a single image
async def process_image(image_path, idx, total, semaphore):
    async with semaphore:  # Limit concurrent requests
        try:
            # Read and encode the image
            with open(image_path, "rb") as image_file:
                base64_image = base64.b64encode(image_file.read()).decode('utf-8')
            
            # Get prediction from API
            response = await client.chat.completions.create(
                model=deployment,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": prompt
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                }
                            }
                        ],
                    }
                ],
            )
            
            # Parse the response
            prediction = response.choices[0].message.content
            
            # Create JSON filename (same as image but with .json extension)
            json_filename = image_path.stem + '.json'
            json_path = os.path.join(output_dir, json_filename)
            
            # Save individual prediction to JSON file
            result = {
                "image_name": image_path.name,
                "prediction": prediction,
                "status": "success"
            }
            
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, indent=2, ensure_ascii=False)
            
            print(f"[{idx+1}/{total}] ✓ Processed: {image_path.name}")
            return True
            
        except Exception as e:
            # Save error to JSON file
            json_filename = image_path.stem + '.json'
            json_path = os.path.join(output_dir, json_filename)
            
            result = {
                "image_name": image_path.name,
                "error": str(e),
                "status": "failed"
            }
            
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, indent=2, ensure_ascii=False)
            
            print(f"[{idx+1}/{total}] ✗ Failed: {image_path.name} - {str(e)}")
            return False

# Main async function
async def process_all_images():
    # Limit concurrent requests to avoid rate limits (adjust as needed)
    semaphore = asyncio.Semaphore(10)
    
    # Create tasks for all images
    tasks = [
        process_image(img_path, idx, len(image_files), semaphore)
        for idx, img_path in enumerate(image_files)
    ]
    
    # Process all images concurrently
    results = await asyncio.gather(*tasks)
    
    success_count = sum(results)
    failed_count = len(results) - success_count
    
    print(f"\nProcessing complete!")
    print(f"Output directory: {output_dir}")
    print(f"Successfully processed: {success_count}/{len(image_files)}")
    print(f"Failed: {failed_count}/{len(image_files)}")

# Run the async processing
await process_all_images()

Found 990 images to process
[10/990] ✓ Processed: 172607979_4_pg1.png
[6/990] ✓ Processed: 172576446_2_pg26.png
[1/990] ✓ Processed: 172427893_3_pg11.png
[4/990] ✓ Processed: 172566189_1_pg10.png
[3/990] ✓ Processed: 172561841_pg1.png
[5/990] ✓ Processed: 172571502_1_pg21.png
[8/990] ✓ Processed: 172585685_3_pg1.png
[9/990] ✓ Processed: 172589759_3_pg6.png
[7/990] ✓ Processed: 172579986_2_pg31.png
[11/990] ✓ Processed: 172610467_2_pg21.png
[14/990] ✓ Processed: 172615659_4_pg18.png
[2/990] ✓ Processed: 172448470_3_pg15.png
[13/990] ✓ Processed: 172615270_3_pg24.png
[16/990] ✓ Processed: 172655019_3_pg11.png
[15/990] ✓ Processed: 172653968_pg20.png
[22/990] ✓ Processed: 172677667_2_pg30.png
[20/990] ✓ Processed: 172675384_1_pg13.png
[19/990] ✓ Processed: 172674674_2_pg26.png
[12/990] ✓ Processed: 172611860_2_pg24.png
[23/990] ✓ Processed: 172679241_1_pg25.png
[18/990] ✓ Processed: 172658339_1_pg46.png
[24/990] ✓ Processed: 172679320_3_pg18.png
[21/990] ✓ Processed: 172676445_3_pg6.png
[

In [2]:
!pip install openai

     -------------------------------------- 387.5/387.5 kB 1.3 MB/s eta 0:00:00
     -------------------------------------- 80.9/80.9 kB 375.8 kB/s eta 0:00:00
     ------------------------------------ 387.1/387.1 kB 189.9 kB/s eta 0:00:00
     ------------------------------------ 386.9/386.9 kB 207.9 kB/s eta 0:00:00
     ------------------------------------ 386.9/386.9 kB 156.6 kB/s eta 0:00:00
     ------------------------------------ 386.9/386.9 kB 217.1 kB/s eta 0:00:00
     ------------------------------------ 383.7/383.7 kB 178.3 kB/s eta 0:00:00
     ------------------------------------ 383.7/383.7 kB 132.8 kB/s eta 0:00:00
     ------------------------------------ 383.5/383.5 kB 197.5 kB/s eta 0:00:00
     ------------------------------------ 383.0/383.0 kB 174.1 kB/s eta 0:00:00
     ------------------------------------ 378.9/378.9 kB 159.5 kB/s eta 0:00:00
     ------------------------------------ 378.9/378.9 kB 216.5 kB/s eta 0:00:00
     -----------------------------------

In [8]:
print(completion.choices[0].message.content)

{
  "fields": {
    "dealer_name": "SHIVAM MOTORS",
    "model_name": "TRACTOR JEETO STRONG 4WD 12F + 3R",
    "horse_power": 24,
    "asset_cost": 549000
  }
}


NameError: name 'image_files' is not defined